# 0. Problem
## 1141. User Activity for the Past 30 Days I — Easy
For each day in the inclusive 30-day window `2019-06-28` through `2019-07-27`, count distinct active users.

Official: https://leetcode.com/problems/user-activity-for-the-past-30-days-i/

# 1. Setup

In [ ]:
import pandas as pd
activity_rows=[(1,1,"2019-07-20","open_session"),(1,1,"2019-07-20","scroll_down"),(1,1,"2019-07-20","end_session"),(2,4,"2019-07-20","open_session"),(2,4,"2019-07-21","send_message"),(2,4,"2019-07-21","end_session"),(3,2,"2019-06-27","open_session")]
activity_pd=pd.DataFrame(activity_rows,columns=["user_id","session_id","activity_date","activity_type"])
activity_pd["activity_date"]=pd.to_datetime(activity_pd["activity_date"])
activity_pd

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark=SparkSession.builder.getOrCreate()
activity_spark=spark.createDataFrame(activity_rows,["user_id","session_id","activity_date","activity_type"]).withColumn("activity_date",F.to_date("activity_date"))
activity_spark.createOrReplaceTempView("Activity")

# 2. SQL Solution

In [ ]:
sql_result=spark.sql("""
SELECT activity_date AS day,COUNT(DISTINCT user_id) AS active_users
FROM Activity
WHERE activity_date BETWEEN DATE('2019-06-28') AND DATE('2019-07-27')
GROUP BY activity_date
ORDER BY day
""")
sql_result.show(truncate=False)

# 3. pandas Solution

In [ ]:
start=pd.Timestamp("2019-06-28"); end=pd.Timestamp("2019-07-27")
result_pd=(activity_pd.loc[activity_pd["activity_date"].between(start,end)].groupby("activity_date",as_index=False).agg(active_users=("user_id","nunique")).rename(columns={"activity_date":"day"}).sort_values("day").reset_index(drop=True))
result_pd

# 4. PySpark Solution

In [ ]:
result_spark=(activity_spark.filter(F.col("activity_date").between(F.to_date(F.lit("2019-06-28")),F.to_date(F.lit("2019-07-27")))).groupBy("activity_date").agg(F.countDistinct("user_id").alias("active_users")).select(F.col("activity_date").alias("day"),"active_users").orderBy("day"))
result_spark.show(truncate=False)

# 5. Pattern Mapping
| Concept | SQL | pandas | PySpark |
|---|---|---|---|
| date window | `BETWEEN` | `.between()` | `.between()` |
| daily unique users | `COUNT(DISTINCT)` | `.nunique()` | `F.countDistinct()` |

# 6. Muscle-Memory Round

พิมพ์ใหม่เองโดยไม่ copy คำตอบด้านบน

In [ ]:
# MUSCLE MEMORY — SQL
# Rebuild using temp view(s): Activity

In [ ]:
# MUSCLE MEMORY — PANDAS
# Rebuild using: activity_pd

In [ ]:
# MUSCLE MEMORY — PYSPARK
# Rebuild using: activity_spark